In [1]:
!pip install ultralytics supervision roboflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.4/217.4 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.5/169.5 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 91.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 132.6 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11


In [2]:
import cv2
import numpy as np
from ultralytics import YOLO
import supervision as sv
from google.colab import drive, files
import os, shutil

drive.mount('/content/drive')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Mounted at /content/drive


In [3]:
# Modèle 1 : TON modèle → détecte le métro (train)
model_metro = YOLO("/content/drive/MyDrive/metro_best.pt")

# Modèle 2 : COCO pré-entraîné → détecte les personnes (classe 0 = person)
model_coco  = YOLO("yolov8n.pt")   # téléchargé automatiquement, aucun entraînement

CLASS_NAMES = model_metro.names
METRO_ID    = [k for k,v in CLASS_NAMES.items() if v == 'train'][0]

print(f"Classes ton modèle : {CLASS_NAMES}")
print(f"METRO_ID = {METRO_ID}")
print("Modèle COCO chargé — détecte 80 classes dont 'person' (ID=0)")

Classes ton modèle : {0: 'in', 1: 'out', 2: 'train'}
METRO_ID = 2
Modèle COCO chargé — détecte 80 classes dont 'person' (ID=0)


In [4]:
class MetroBoxTracker:
    """
    Stabilise la bounding box du metro sur plusieurs frames
    pour eviter les sauts brusques de position.
    """
    def __init__(self, smoothing=8):
        self.history   = []       # historique des dernières boxes
        self.smoothing = smoothing

    def update(self, new_box):
        if new_box is not None:
            self.history.append(new_box)
        if len(self.history) > self.smoothing:
            self.history.pop(0)

    def get_stable_box(self):
        if not self.history:
            return None
        # Moyenne sur les N dernières boxes
        arr = np.array(self.history)
        return tuple(np.mean(arr, axis=0).astype(int))

print("MetroBoxTracker prêt")

MetroBoxTracker prêt


In [5]:
class PersonCounter:
    """
    Gere le comptage IN/OUT avec :
    - Vote majoritaire de classe sur toutes les frames
    - Cooldown : une personne ne peut pas etre comptee 2x en moins de N frames
    - Historique des etats (dans/hors metro) pour detecter la transition
    """
    def __init__(self, cooldown_frames=15):
        self.count_in        = 0
        self.count_out       = 0
        self.previous_state  = {}    # track_id -> True/False (dans/hors)
        self.class_votes     = {}    # track_id -> [class_ids]
        self.last_counted    = {}    # track_id -> frame_number
        self.cooldown        = cooldown_frames
        self.events          = []    # log de chaque evenement

    def update(self, track_id, class_id, is_inside, frame_num, timestamp):
        # Voter pour la classe
        if track_id not in self.class_votes:
            self.class_votes[track_id] = []
        self.class_votes[track_id].append(class_id)

        prev = self.previous_state.get(track_id, None)

        if prev is not None and prev != is_inside:
            # Transition détectée
            # Vérifier le cooldown
            last = self.last_counted.get(track_id, -999)
            if frame_num - last < self.cooldown:
                self.previous_state[track_id] = is_inside
                return  # trop tôt, ignorer

            # Classe majoritaire
            votes      = self.class_votes.get(track_id, [class_id])
            main_class = max(set(votes), key=votes.count)

            if is_inside:   # vient d'entrer dans le metro → IN
                self.count_in += 1
                self.last_counted[track_id] = frame_num
                self.events.append({
                    "frame": frame_num, "timestamp": timestamp,
                    "event": "IN", "track_id": int(track_id),
                    "total_in": self.count_in, "total_out": self.count_out
                })
                print(f"  IN  | ID:{track_id} | frame:{frame_num} | votes:{len(votes)}")

            else:           # vient de sortir du metro → OUT
                self.count_out += 1
                self.last_counted[track_id] = frame_num
                self.events.append({
                    "frame": frame_num, "timestamp": timestamp,
                    "event": "OUT", "track_id": int(track_id),
                    "total_in": self.count_in, "total_out": self.count_out
                })
                print(f"  OUT | ID:{track_id} | frame:{frame_num} | votes:{len(votes)}")

        self.previous_state[track_id] = is_inside

print("PersonCounter prêt")

PersonCounter prêt


In [7]:
# Ajoute cette cellule de diagnostic AVANT le code principal
# Pour voir exactement ce que COCO voit sur une seule frame

cap = cv2.VideoCapture(list(uploaded.keys())[0])
ret, frame = cap.read()
cap.release()

# Tester avec conf très bas pour tout voir
results = model_coco(frame, verbose=True, conf=0.01)[0]

print("\nTout ce que COCO détecte (conf > 0.01) :")
for box, cls, conf in zip(results.boxes.xyxy, results.boxes.cls, results.boxes.conf):
    print(f"  Classe {int(cls):2d} ({model_coco.names[int(cls)]:15s}) | confiance : {conf:.3f}")


0: 640x384 85 persons, 9.1ms
Speed: 3.9ms preprocess, 9.1ms inference, 1.9ms postprocess per image at shape (1, 3, 640, 384)

Tout ce que COCO détecte (conf > 0.01) :
  Classe  0 (person         ) | confiance : 0.789
  Classe  0 (person         ) | confiance : 0.765
  Classe  0 (person         ) | confiance : 0.673
  Classe  0 (person         ) | confiance : 0.668
  Classe  0 (person         ) | confiance : 0.638
  Classe  0 (person         ) | confiance : 0.627
  Classe  0 (person         ) | confiance : 0.614
  Classe  0 (person         ) | confiance : 0.520
  Classe  0 (person         ) | confiance : 0.460
  Classe  0 (person         ) | confiance : 0.440
  Classe  0 (person         ) | confiance : 0.371
  Classe  0 (person         ) | confiance : 0.369
  Classe  0 (person         ) | confiance : 0.131
  Classe  0 (person         ) | confiance : 0.115
  Classe  0 (person         ) | confiance : 0.108
  Classe  0 (person         ) | confiance : 0.102
  Classe  0 (person         ) | 

In [11]:
import csv

uploaded = files.upload()

for video_path in uploaded.keys():
    print(f"\n{'='*50}")
    print(f"Traitement : {video_path}")
    print(f"{'='*50}")

    output_path  = f"resultat_{video_path}"
    cap    = cv2.VideoCapture(video_path)
    W      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    FPS    = cap.get(cv2.CAP_PROP_FPS)
    writer = cv2.VideoWriter(
        output_path, cv2.VideoWriter_fourcc(*"mp4v"), FPS, (W, H)
    )

    tracker       = sv.ByteTrack()
    metro_tracker = MetroBoxTracker(smoothing=8)
    counter       = PersonCounter(cooldown_frames=15)
    frame_num     = 0

    # ── CORRECTION : garder la dernière détection entre frames ──────
    last_det_persons = sv.Detections.empty()   # ← nouveau

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_num += 1
        timestamp = round(frame_num / FPS, 2)

        # ── MODELE 1 : métro (toutes les frames) ────────────────────
        results_metro = model_metro(frame, verbose=False, conf=0.4)[0]
        det_metro     = sv.Detections.from_ultralytics(results_metro)
        metro_mask    = det_metro.class_id == METRO_ID

        if metro_mask.any():
            best_idx = np.argmax(det_metro.confidence[metro_mask])
            boxes_m  = det_metro.xyxy[metro_mask]
            mx1, my1, mx2, my2 = boxes_m[best_idx]
            mw = mx2 - mx1
            mh = my2 - my1
            new_box = (
                int(mx1 + mw * 0.08),
                int(my1 + mh * 0.08),
                int(mx2 - mw * 0.08),
                int(my2 - mh * 0.08)
            )
            metro_tracker.update(new_box)
        else:
            metro_tracker.update(None)

        # ── MODELE 2 COCO : personnes (1 frame sur 2) ───────────────
        # CORRECTION : si on ne détecte pas, on passe la DERNIERE détection
        # au tracker (pas une liste vide) → le tracking reste stable
        if frame_num % 2 == 0:
            results_coco     = model_coco(frame, verbose=False,
                                          conf=0.25, classes=[0])[0]
            last_det_persons = sv.Detections.from_ultralytics(results_coco)

        # Toujours passer au tracker — avec la détection actuelle ou la dernière
        det_persons = tracker.update_with_detections(last_det_persons)

        # ── Box métro stabilisée ─────────────────────────────────────
        metro_box = metro_tracker.get_stable_box()
        if metro_box is None:
            writer.write(frame)
            continue

        mx1, my1, mx2, my2 = metro_box

        # ── Comptage par transition ──────────────────────────────────
        for box, track_id in zip(det_persons.xyxy, det_persons.tracker_id):
            if track_id is None:
                continue

            px1, py1, px2, py2 = box
            pcx = int((px1 + px2) / 2)
            pcy = int((py1 + py2) / 2)

            is_inside = (mx1 <= pcx <= mx2) and (my1 <= pcy <= my2)

            counter.update(
                track_id  = track_id,
                class_id  = 0,
                is_inside = is_inside,
                frame_num = frame_num,
                timestamp = f"{timestamp}s"
            )

            color = (0, 255, 0) if is_inside else (200, 200, 200)
            state = "DANS" if is_inside else "HORS"
            cv2.rectangle(frame, (int(px1), int(py1)),
                          (int(px2), int(py2)), color, 2)
            cv2.putText(frame, f"ID:{track_id} {state}",
                        (int(px1), int(py1) - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)

        # ── Dessiner métro ───────────────────────────────────────────
        cv2.rectangle(frame, (mx1, my1), (mx2, my2), (255, 150, 0), 2)
        cv2.putText(frame, "METRO", (mx1, my1 - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 150, 0), 2)
        cv2.line(frame, (mx1, my1), (mx1, my2), (100, 100, 255), 2)
        cv2.line(frame, (mx2, my1), (mx2, my2), (100, 100, 255), 2)

        # ── Compteurs ────────────────────────────────────────────────
        cv2.rectangle(frame, (0, 0), (280, 75), (0, 0, 0), -1)
        cv2.putText(frame, f"IN  (montes)   : {counter.count_in}",
                    (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        cv2.putText(frame, f"OUT (descendus): {counter.count_out}",
                    (10, 62), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

        writer.write(frame)

    cap.release()
    writer.release()
    print(f"\nTerminé → IN={counter.count_in}  OUT={counter.count_out}")

    if counter.events:
        csv_path = f"/content/drive/MyDrive/comptage_{video_path}.csv"
        with open(csv_path, "w", newline="") as f:
            writer_csv = csv.DictWriter(f, fieldnames=counter.events[0].keys())
            writer_csv.writeheader()
            writer_csv.writerows(counter.events)
        print(f"CSV sauvegardé : {csv_path}")

    shutil.copy(output_path, f"/content/drive/MyDrive/{output_path}")
    files.download(output_path)

Saving Park_kultury.MP4 to Park_kultury.MP4

Traitement : Park_kultury.MP4
  OUT | ID:48 | frame:680 | votes:16
  OUT | ID:49 | frame:696 | votes:14
  OUT | ID:51 | frame:728 | votes:32
  IN  | ID:59 | frame:766 | votes:26
  OUT | ID:64 | frame:789 | votes:27
  OUT | ID:80 | frame:824 | votes:20
  OUT | ID:84 | frame:857 | votes:19
  OUT | ID:99 | frame:924 | votes:10
  OUT | ID:96 | frame:930 | votes:6
  OUT | ID:106 | frame:976 | votes:22

Terminé → IN=1  OUT=9
CSV sauvegardé : /content/drive/MyDrive/comptage_Park_kultury.MP4.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>